In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
torch.manual_seed(42)
X = torch.randn(4,8) # (seq_len, d_model) 4 words , 8 dimensions

In [3]:
# Learned projection metrices 
W_Q = nn.Linear(8,8,bias=False)
W_K  = nn.Linear(8,8,bias=False)
W_V = nn.Linear(8,8,bias=False)

In [4]:
# Project into Q, K, V
Q = W_Q(X)  # (4, 8)
K = W_K(X)  # (4, 8)
V = W_V(X)  # (4, 8)

In [5]:
# sCALED Dot Product attention 
d_k = Q.shape[-1]
scores = Q @ K.T / d_k**0.5
weights  = F.softmax(scores,dim=-1)
output = weights @ V

In [6]:
print(f"Attention weights:\n{weights.detach().round(decimals=3)}")
print(f"Output shape: {output.shape}")

Attention weights:
tensor([[0.3330, 0.2170, 0.1850, 0.2650],
        [0.1960, 0.2570, 0.2310, 0.3160],
        [0.3560, 0.2550, 0.2570, 0.1320],
        [0.1720, 0.2680, 0.3670, 0.1930]])
Output shape: torch.Size([4, 8])


In [7]:
multihead_attn = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)

In [8]:
X_batched = X.unsqueeze(0)  # (1, 4, 8) — batch=1, seq=4, dim=8

In [9]:
attn_output, attn_weights = multihead_attn(X_batched, X_batched, X_batched)

In [10]:
print(f"Output shape: {attn_output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nAttention weights:\n{attn_weights.detach().squeeze().round(decimals=3)}")

Output shape: torch.Size([1, 4, 8])
Attention weights shape: torch.Size([1, 4, 4])

Attention weights:
tensor([[0.2090, 0.2450, 0.2990, 0.2480],
        [0.2420, 0.3550, 0.1690, 0.2330],
        [0.0740, 0.2220, 0.4020, 0.3020],
        [0.1730, 0.3080, 0.2480, 0.2710]])


# Transformer encoder

In [17]:
class TransformerBlock(nn.Module):
    def __init__(self,d_model=8,num_heads=2,ff_dim=32,dropout=0.1):
        super().__init__()

        #Multi-head self attention
        self.attention = nn.MultiheadAttention(d_model,  num_heads, batch_first=True,dropout=dropout)

        # FF -> feed dorward network - two linear layers with ReeLU
        self.ff = nn.Sequential(
            nn.Linear(d_model,ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim,d_model)
        )

        # Normalizsation - like batch norma but per sequence 
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        # Self attention + Residual ++ layernorm 
        attn_out, _ = self.attention(x,x,x)
        x = self.norm1(x+self.dropout(attn_out))

        ff_out = self.ff(x)
        x = self.norm2(x+self.dropout(ff_out))

        return x

In [18]:
block = TransformerBlock()
out = block(X_batched)
print(f"Input shape:  {X_batched.shape}")
print(f"Output shape: {out.shape}")
print("Shape preserved — transformer block works correctly")

Input shape:  torch.Size([1, 4, 8])
Output shape: torch.Size([1, 4, 8])
Shape preserved — transformer block works correctly


## Summary

**What I built:** Self-attention in PyTorch — manual Q/K/V implementation, 
nn.MultiheadAttention, and a full TransformerBlock with residual connections 
and LayerNorm.

**Key concepts:**
- nn.Linear vs np.random: Linear has requires_grad=True — weights are learnable
- Self-attention: Q=K=V=same input — sequence attends to itself
- Cross-attention: Q from one sequence, K/V from another
- nn.MultiheadAttention: splits d_model into num_heads parallel attention heads
- batch_first=True: input shape (batch, seq, dim) not (seq, batch, dim)
- Residual connection: x = norm(x + attn_out) — gradients flow directly, prevents vanishing
- LayerNorm: normalizes per sequence position, not per batch (unlike BatchNorm)
- TransformerBlock = MultiheadAttention + Add&Norm + FeedForward + Add&Norm

**Why residual connections matter:**
- Gradients flow directly to early layers — no vanishing gradient in deep stacks
- Early in training: attention output is garbage, residual passes input unchanged
- BERT stacks 12 of these blocks — residuals make this trainable

**Next:** Week 6 review — Day 42